# Zakliata reč

V tomto notebooku prejdeš od prekladu slovo po slove k malému modelu.
Postupuj v štyroch krokoch:

1. Prezri si páry viet a pomenuj chybu slovníkového prekladu
2. Zmeraj slovník na odložených pároch, ktoré pri učení nevidel
3. Doplň tri tréningové funkcie, natrénuj model a porovnaj chyby
4. Zmeň jednu vec a vysvetli, čo sa zlepšilo alebo zhoršilo

Na prvé dva kroky stačí CPU. Pre tretí krok vyber v Colabe
**Runtime → Change runtime type → T4 GPU** ešte pred úpravou `train.py`.
Kontroly funkcií bežia na CPU. Tréning spustíš samostatne.

Nahraj `train_data.zip`, `test_data.zip` a `starter_kit.zip` cez ikonu
priečinka vľavo. Mimo Colabu rozbaľ archívy ručne
do priečinka s notebookom. Testovacie preklady si nechaj až na odovzdanie.


In [ ]:
from pathlib import Path
import zipfile

for name in ("train_data.zip", "test_data.zip", "starter_kit.zip"):
    if Path(name).exists():
        with zipfile.ZipFile(name) as archive:
            for member in archive.namelist():
                if name == "starter_kit.zip" and Path(member).exists():
                    print(f"Zachovávam {member}, aby sa tvoje úpravy neprepísali.")
                else:
                    archive.extract(member, ".")
for name in ("data/train.csv", "data/test.csv", "starter.py", "train.py"):
    assert Path(name).exists(), f"Chýba {name}. Nahraj a rozbaľ príslušný archív."
print("Dáta a zdrojové súbory sú pripravené.")

## 1. Prezri si dáta

Slovník v `starter.py` sa z dát naučí, ktoré slová si v pároch zodpovedajú.
Každé slovo potom prekladá samostatne. Nevníma kontext a nemení poradie slov.

Prečítaj si niekoľko párov. Všimni si opakujúce sa korene, koncovky
a poradie slov. Zapíš si jednu vlastnosť viet, ktorú samostatný preklad
slov nedokáže zachytiť. Zatiaľ nemusíš rozlúštiť celú gramatiku.


In [ ]:
from starter import load_pairs, split_pairs, fit, evaluate, write_submission

pairs = load_pairs()
print(f"Počet párov: {len(pairs)}")
for zak, sk in pairs[:5]:
    print(f"\nZak: {zak}\nSk:  {sk}")

## 2. Zmeraj slovníkový preklad

Bunka nižšie odloží 500 celých párov na validáciu. Slovník sa učí iba zo zvyšku.
Oba smery jedného páru musia zostať v tej istej časti dát.

chrF meria podobnosť znakových skupín s referenčným prekladom.
Vyššie skóre je lepšie. Validačné skóre slúži na porovnanie pokusov,
nie je to výsledok na súťažnom teste.


In [ ]:
%pip -q install sacrebleu==2.5.1 tokenizers==0.22.1

In [ ]:
train_pairs, dev_pairs = split_pairs(pairs, dev_size=500, seed=0)
print(f"Učenie: {len(train_pairs)} párov, validácia: {len(dev_pairs)} párov")
tables = fit(train_pairs)
baseline_scores = evaluate(dev_pairs, tables)
print("Zapíš si tieto dve skóre ako východiskový výsledok.")

Vyber jednu z vypísaných chýb. Je problém v neznámom slove,
v koncovke, v poradí slov alebo vo vynechanom slove?
Napíš, akú informáciu by prekladač potreboval na jej opravu.


## 3. Doplň tréning modelu

`train.py` je kostra malého Transformera typu enkodér-dekodér.
Enkodér spracuje vstupnú vetu. Dekodér postupne predpovedá tokeny prekladu.
Tokenizér delí text na časti slov. Značka na začiatku vstupu určuje cieľový jazyk.
Architektúra, tokenizér, dekódovanie a zápis výsledkov sú pripravené.
Tréning sa spustí až po doplnení troch funkcií.

Otvor `train.py` v editore súborov vľavo a postupne doplň nasledujúce funkcie.
Upravený súbor si priebežne sťahuj aj do počítača, Colab lokálne súbory
neuchováva natrvalo.

BOS označuje začiatok vety a EOS jej koniec. Výplňové tokeny vyrovnávajú
dĺžky viet v dávke.


1. **`make_examples`**: z každého trénovacieho páru vytvor príklady pre oba
   smery a označ cieľový jazyk. Zamysli sa, prečo páry rozdeľujeme na trénovacie a validačné ešte pred týmto krokom
2. **`train_batch`**: zostav vstup a správne odpovede dekodéra, spočítaj
   stratu (loss) bez započítania výplňových tokenov a vykonaj krok tréningu. Čo by sa model naučil, keby
   na vstupe dekodéra dostal ten istý token, ktorý má práve predpovedať?
3. **`select_checkpoint`**: vyber stav modelu (checkpoint) s najvyšším priemerom
   validačného chrF oboch smerov a ulož jeho váhy. Prečo nestačí vziať model
   z poslednej epochy alebo ten s najnižšou tréningovou stratou?

Komentáre vo funkciách opisujú vstupy a očakávané výstupy.
Po každej úprave súbor ulož a spusti nasledujúcu bunku. Kontroly bežia na CPU,
vypíšu výsledok kontroly každej z troch funkcií a nevytvoria preklady na odovzdanie.
Úspešná kontrola ešte neznamená, že model dobre prekladá.


In [ ]:
import subprocess
import sys

check = subprocess.run([sys.executable, "train.py", "--check"],
                       capture_output=True, text=True)
print(check.stdout)
if check.returncode:
    print(check.stderr)
    print("Doplň označené funkcie v train.py, ulož súbor a zopakuj kontrolu.")
NEURAL_READY = check.returncode == 0

### Spusti tréning

Model používa rovnakých 500 validačných párov ako slovník. Tokenizér sa učí
iba z trénovacej časti. Pri tréningu sleduj stratu aj validačné chrF.
Pred spustením odhadni, ktorý druh chyby by mal model opraviť.

Ak kontrola hlási 3/3 a máš zapnuté GPU T4, nastav `RUN_TRAINING = True`.
Skript má predvolený časový limit tréningu 20 minút (`--minutes`).
Kontroluje ho po každej epoche. Na záverečné vyhodnotenie si nechaj čas navyše.
V prvom pokuse môžeš nastaviť menšie `--max_epochs`,
aby si si najprv overil, že celý postup funguje.


In [ ]:
RUN_TRAINING = False  # Zapni po doplnení troch funkcií a kontrole 3/3.

if RUN_TRAINING:
    assert NEURAL_READY, "Najprv doplň train.py a zopakuj kontrolu troch funkcií."
    import subprocess
    import sys
    import torch
    assert torch.cuda.is_available(), "V Colabe najprv zapni GPU T4."
    subprocess.run([
        sys.executable, "train.py", "--data", "data", "--out", "model_run.csv",
        "--d", "256", "--layers", "3", "--dropout", "0.3",
        "--max_epochs", "80", "--minutes", "20",
    ], check=True)
else:
    print("Tréning je vypnutý. Najprv doplň tri funkcie a spusti ich kontrolu.")

In [ ]:
import csv
import json

if RUN_TRAINING:
    metrics = json.loads(Path("model_run.metrics.json").read_text())
    print(json.dumps(metrics, indent=2, ensure_ascii=False))
    with open("model_run.dev.csv", encoding="utf-8", newline="") as f:
        errors = list(csv.DictReader(f))
    for st in ("1", "2"):
        print(f"\nPodúloha {st}")
        for row in [r for r in errors if r["subtaskID"] == st][:3]:
            print(f"Vstup: {row['input']}\nReferencia: {row['reference']}\nModel: {row['answer']}\n")

Porovnaj obe skóre chrF so slovníkom. Pozri sa aj na rovnaké vety
z kroku 2. Opravil model chybu, ktorú si predpovedal? Pribudla nejaká nová chyba?
Správny preklad sa môže líšiť od jediného referenčného znenia,
preto si pri hodnotení všímaj aj význam.


## 4. Zmeň jednu vec a porovnaj výsledok

Nechaj rovnaké validačné páry a zmeň iba jedno nastavenie.
Môžeš napríklad porovnať dropout 0.3 a 0.1 pri rovnakom počte epoch.
Pred pokusom si zapíš očakávaný výsledok. Nový výstup pomenuj inak,
napríklad `model_dropout01.csv`, aby si mohol porovnať aj chyby.
Ak porovnávaš nastavenia modelu, nastav dosť času,
aby oba pokusy stihli rovnaký počet epoch.

| Pokus | Jediná zmena | chrF zak → sk | chrF sk → zak | Pozorovaná chyba |
|---|---|---|---|---|
| Slovník | Východiskový výsledok | doplň | doplň | doplň |
| Model | Preklad celej vety v kontexte | doplň | doplň | doplň |
| Upravený model | doplň | doplň | doplň | doplň |

Ku každému modelovému pokusu vzniknú aj súbory `.metrics.json` a `.dev.csv`.
Zapíš si, čo výsledok naznačuje a čo sa z jedného pokusu ešte nedá povedať.
Porovnávaj pokusy na validácii. Súťažný test nepoužívaj na výber nastavení.

## Priprav odovzdanie

Pri vypnutom tréningu bunka nižšie vytvorí `submission.csv` zo slovníka.
Ak si trénoval model, bunka slovníkové odovzdanie neprepíše. Porovnaj model
so slovníkom na validácii a vyber lepší pokus. Ak odovzdávaš model, použi
jeho súbor `model_run.csv` premenovaný na `submission.csv`.

Spolu s prekladmi odovzdaj aj kód, ktorý ich vyrobil. Pri modeli prilož
použitý príkaz, závislosti a seed. Zdrojový kód môže mať najviac 35 KB.
Odovzdať môžeš aj samotné `.py` súbory. Z notebooku odstráň výstupy.
Ak používaš doplnený `train.py`, odovzdaj aj `starter.py`. Váhy neprikladaj.
Tabuľku pokusov a rozbor troch chýb si nechaj ako záznam
svojho postupu. Nebodujú sa.


In [ ]:
if RUN_TRAINING:
    print("Modelové preklady sú v model_run.csv. Pred odovzdaním vyber lepší pokus podľa validácie.")
else:
    write_submission(fit(pairs))